# autodiff — plots

Run this notebook **once `make test` is green**. It uses *your* `autodiff` package to
display three things:

1. a function and its tangents — a visual check of `derivative`;
2. the error of four ways of computing a derivative, as a function of the step `h`;
3. the cost of a gradient, forward mode vs reverse mode, when the dimension grows.

In [ ]:
import cmath
import math
import sys
import time

import matplotlib.pyplot as plt

sys.path.insert(0, "..")          # the notebook lives in notebooks/, the package one level up
from autodiff import derivative, exp, gradient_forward, gradient_reverse, sin

## 1. A function and its tangents

The tangent at `a` is the line `y = f(a) + f'(a)(x − a)`: the best linear approximation
of `f` near `a`. If your `derivative` is right, each line touches the curve without
crossing it locally.

In [ ]:
def f(x):
    return sin(x) * exp(-0.2 * x) + 0.1 * x

xs = [i / 50 for i in range(0, 501)]                 # [0, 10]
plt.figure(figsize=(8, 4))
plt.plot(xs, [f(x) for x in xs], label="f")
for a in (1.0, 3.5, 6.0, 8.5):
    slope = derivative(f, a)
    window = [a + t / 10 for t in range(-12, 13)]
    plt.plot(window, [f(a) + slope * (x - a) for x in window], "--")
    plt.plot([a], [f(a)], "o", color="black")
plt.title("f and four tangents computed with dual numbers")
plt.legend()
plt.show()

## 2. Four ways to compute a derivative

We compute `f'(1)` for `f(x) = x·eˣ` (exact value `2e`) and plot the error against `h`:

- **forward difference** `(f(x+h) − f(x))/h`: truncation error `O(h)`;
- **central difference** `(f(x+h) − f(x−h))/2h`: truncation error `O(h²)`;
- **complex step** `Im f(x + ih)/h` (see the lecture's bonus): `O(h²)`, but no subtraction;
- **dual numbers**: no `h` at all.

Predict the shape of each curve before running the cell!

In [ ]:
exact = 2 * math.e

def g(x):
    return x * math.exp(x)

def g_complex(z):
    return z * cmath.exp(z)

hs = [10.0 ** (-k / 2) for k in range(2, 41)]        # 1e-1 ... 1e-20
floor = 1e-17                                          # to show a zero error on a log scale
forward = [max(abs((g(1 + h) - g(1)) / h - exact), floor) for h in hs]
central = [max(abs((g(1 + h) - g(1 - h)) / (2 * h) - exact), floor) for h in hs]
complex_step = [max(abs(g_complex(1 + 1j * h).imag / h - exact), floor) for h in hs]
dual = max(abs(derivative(lambda t: t * exp(t), 1.0) - exact), floor)

plt.figure(figsize=(8, 5))
plt.loglog(hs, forward, "o-", label="forward difference")
plt.loglog(hs, central, "s-", label="central difference")
plt.loglog(hs, complex_step, "^-", label="complex step")
plt.loglog(hs, [dual] * len(hs), "k--", label="dual numbers (no h)")
plt.gca().invert_xaxis()
plt.xlabel("step h")
plt.ylabel("absolute error on f'(1)")
plt.legend()
plt.show()

**Questions.**
- Why do the finite-difference curves go back up when `h` becomes very small?
- Read the best `h` for each difference formula. Compare with `√ε ≈ 1e-8` and `ε^(1/3) ≈ 6e-6`.
- Why does the complex step not go back up?
- Why is the dual-number error at machine precision whatever you do?

## 3. The cost of a gradient: forward vs reverse

Rosenbrock in dimension `n`. Forward mode needs **one sweep per variable**; reverse mode
needs **one backward sweep** whatever `n`. We plot the time of a gradient divided by the
time of a plain evaluation of the function.

In [ ]:
def rosenbrock(x):
    return sum(100 * (x[i + 1] - x[i] ** 2) ** 2 + (1 - x[i]) ** 2 for i in range(len(x) - 1))

def timed(fun, repeat=3):
    best = float("inf")
    for _ in range(repeat):
        start = time.perf_counter()
        fun()
        best = min(best, time.perf_counter() - start)
    return best

dims = [2, 5, 10, 20, 50, 100, 200]
ratio_forward, ratio_reverse = [], []
for n in dims:
    point = [0.5] * n
    t_value = timed(lambda: rosenbrock(point))
    ratio_forward.append(timed(lambda: gradient_forward(rosenbrock, point)) / t_value)
    ratio_reverse.append(timed(lambda: gradient_reverse(rosenbrock, point)) / t_value)

plt.figure(figsize=(8, 5))
plt.loglog(dims, ratio_forward, "o-", label="forward mode")
plt.loglog(dims, ratio_reverse, "s-", label="reverse mode")
plt.xlabel("dimension n")
plt.ylabel("time(gradient) / time(value)")
plt.legend()
plt.show()

**Questions.**
- What is the slope of the forward-mode curve? Why?
- The reverse-mode ratio stays roughly constant: what does it cost in exchange? (Think about memory.)
- That constant (≈ 50 here) is large because every `Var` is a Python object. Frameworks written in C/C++ bring it down to a few units — but the *shape* of the curves is the same.
- A neural network has millions of parameters and one scalar loss. Which mode do ML frameworks use?